# TDDA: Test-Driven Data Analysis

[TDDA](https://github.com/tdda/tdda) verwendet Dateieingaben (wie NumPy-Arrays oder Pandas DataFrames) und eine Reihe von Einschränkungen (engl.: _constraints_), die als JSON-Datei gespeichert werden.

* [Reference Test](https://tdda.readthedocs.io/en/latest/referencetest.html) unterstützt die Erstellung von Regressionstests, die entweder auf `unittest` oder `pytest` basieren.
* [Constraints](https://tdda.readthedocs.io/en/v1.0.30/constraints.html) wird verwendet, um Constraints aus einem (Pandas)-DataFrame zu ermitteln, sie als JSON auszuschreiben und zu überprüfen, ob Datensätze die Constraints in der Constraints-Datei erfüllen. Es unterstützt auch Tabellen in einer Vielzahl von relationalen Datenbanken.
* [Rexpy](https://tdda.readthedocs.io/en/v1.0.30/rexpy.html) ist ein Werkzeug zur automatischen Ableitung von regulären Ausdrücken aus einer Spalte in einem Pandas DataFrame oder aus einer (Python)-Liste von Beispielen.

TDDA kann auch über die Befehlszeile genutzt werden und kann somit auch von allen, die mit Daten arbeiten, verwendet werden – unabhängig davon, ob sie Python, R, Excel, SQL oder andere Sprachen und Werkzeuge verwenden. 

## 1. Importe

In [1]:
from pathlib import Path

import pandas as pd

from tdda.constraints import detect_df, discover_df, verify_df

In [2]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/kjam/data-cleaning-101/master/data/iot_example.csv",
)

## 2. Daten überprüfen

Mit [pandas.DataFrame.sample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html) lassen wir uns die ersten zehn Datensätze anzeigen:

In [3]:
df.sample(10)

,timestamp,username,temperature,heartrate,build,latest,note
49413,2017-01-21T05:58:08,smithkimberly,22,89,6e5ee2cc-7456-8ea6-5747-abaa556a8082,1,sleep
22613,2017-01-10T12:31:43,ajohnson,14,60,4d28cbd5-d71a-ab3e-e014-1442f39934f3,0,NaN
80838,2017-02-02T20:02:21,williamoliver,10,78,c3a99af0-f0e9-bac0-b3da-f04eec180c82,0,interval
51285,2017-01-21T23:53:12,breannabarron,28,60,9cc0fac1-5abf-d0ad-5c1f-ac7a55ab6d4a,0,test
8673,2017-01-04T23:11:21,nsmith,24,66,e6389298-3aa6-17a5-520c-3c43b2b7e66c,1,interval
135766,2017-02-24T17:59:53,gregory24,19,77,4e3f75d2-1426-40cf-5608-e37ccc4143f4,0,wake
129791,2017-02-22T08:29:55,eavila,10,61,4757267e-5dd7-500d-bbda-6baed3c15e9d,0,update
88058,2017-02-05T17:06:12,salazardeborah,5,84,19f3b095-f691-ddda-291f-ad115876fdb5,1,sleep
57588,2017-01-24T12:20:20,bradleymary,8,75,41e1fd3e-a1e8-c6b0-8ab8-3fecd866cc7f,0,test
143432,2017-02-27T19:36:22,james76,16,81,d5962e7e-ba2a-6a90-6eaa-db584c02d2c0,0,wake


Und mit [pandas.DataFrame.dtypes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html) lassen wir uns die Datentypen für die einzelnen Spalten anzeigen:

In [4]:
df.dtypes

timestamp      object
username       object
temperature     int64
heartrate       int64
build          object
latest          int64
note           object
dtype: object

## 3. Erstellen eines _constraints_-Objekt

Mit `discover_df` kann ein Constraints-Objekt erzeugt werden.

In [5]:
constraints = discover_df(df)

In [6]:
constraints

In [7]:
constraints.fields

Fields([('timestamp', <tdda.constraints.base.FieldConstraints at 0x14baa0ad0>),
        ('username', <tdda.constraints.base.FieldConstraints at 0x1383aead0>),
        ('temperature',
         <tdda.constraints.base.FieldConstraints at 0x1383aefd0>),
        ('heartrate', <tdda.constraints.base.FieldConstraints at 0x1383836f0>),
        ('build', <tdda.constraints.base.FieldConstraints at 0x138383a80>),
        ('latest', <tdda.constraints.base.FieldConstraints at 0x1383f4830>),
        ('note', <tdda.constraints.base.FieldConstraints at 0x1194078a0>)])

## 4. Schreiben der _Constraints_ in eine Datei

In [8]:
with Path.open("../../../data/iot_example.json", "w") as f:
    f.write(constraints.to_json())

Wenn wir uns die Datei genauer betrachten können wir erkennen, dass z.B. für die `timestamp`-Spalte eine Zeichenkette mit 19 Zeichen erwartet wird und `temperature` Integer mit Werten von 5–29 erwartet.

In [9]:
!cat ../../../data/iot_example.json

{
    "creation_metadata": {
        "local_time": "2026-09-15T14:31:16",
        "utc_time": "2026-09-15T12:31:16+00:00",
        "creator": "TDDA 2.2.17",
        "host": "fay.local",
        "user": "veit",
        "n_records": 146397,
        "n_selected": 146397
    },
    "fields": {
        "timestamp": {
            "type": "string",
            "min_length": 19,
            "max_length": 19,
            "max_nulls": 0,
            "no_duplicates": true
        },
        "username": {
            "type": "string",
            "min_length": 3,
            "max_length": 21,
            "max_nulls": 0
        },
        "temperature": {
            "type": "int",
            "min": 5,
            "max": 29,
            "sign": "positive",
            "max_nulls": 0
        },
        "heartrate": {
            "type": "int",
            "min": 60,
            "max": 89,
            "sign": "positive",
            "max_nulls": 0
        },
        "build": {
            "type": "s

Ihr könnt mit `discover_df` auch versuchen, reguläre Ausdrücke zu ermitteln:

In [10]:
constraints = discover_df(df, inc_rex=True)

In [11]:
with Path.open("../../../data/iot_re_example.json", "w") as f:
    f.write(constraints.to_json())

In [12]:
!cat ../../../data/iot_example.json

{
    "creation_metadata": {
        "local_time": "2026-09-15T14:31:16",
        "utc_time": "2026-09-15T12:31:16+00:00",
        "creator": "TDDA 2.2.17",
        "host": "fay.local",
        "user": "veit",
        "n_records": 146397,
        "n_selected": 146397
    },
    "fields": {
        "timestamp": {
            "type": "string",
            "min_length": 19,
            "max_length": 19,
            "max_nulls": 0,
            "no_duplicates": true
        },
        "username": {
            "type": "string",
            "min_length": 3,
            "max_length": 21,
            "max_nulls": 0
        },
        "temperature": {
            "type": "int",
            "min": 5,
            "max": 29,
            "sign": "positive",
            "max_nulls": 0
        },
        "heartrate": {
            "type": "int",
            "min": 60,
            "max": 89,
            "sign": "positive",
            "max_nulls": 0
        },
        "build": {
            "type": "s

## 5. Überprüfen von Dataframes

Hierfür lesen wir zunächst eine neue csv-Datei mit Pandas ein und lassen uns dann zehn Datensätze exemplarisch ausgeben:

In [13]:
new_df = pd.read_csv(
    "https://raw.githubusercontent.com/kjam/data-cleaning-101/master/data/iot_example_with_nulls.csv"
)

new_df.sample(10)

,timestamp,username,temperature,heartrate,build,latest,note
20216,2017-01-09T13:41:53,nwalker,11.0,68,b473c350-ca8f-86f4-06c9-ef1c21530534,NaN,user
130644,2017-02-22T16:41:58,shanemcgee,12.0,83,19c1d63f-0c2c-290d-0f15-707a208d530e,0.0,wake
49847,2017-01-21T10:08:00,sgarrett,NaN,68,c28fe23f-a5f3-c066-cafb-8e01ac02d164,0.0,NaN
139008,2017-02-26T01:05:15,nicolephillips,23.0,82,NaN,1.0,sleep
107991,2017-02-13T16:02:26,murphypamela,9.0,65,79206a21-bafc-c6b4-5238-91546bff1195,1.0,wake
38215,2017-01-16T18:19:39,marcia78,6.0,66,fd1d7e54-e347-6ab8-9bec-1b5e8fed7c54,NaN,sleep
8314,2017-01-04T19:45:11,ywilson,20.0,71,10fa0a5e-1ad4-22d0-d629-feba97ef1c0f,NaN,interval
84211,2017-02-04T04:26:12,robertroy,25.0,69,9f6059e7-5fa9-a4fe-6cf5-3ed3af1e6856,0.0,wake
101636,2017-02-11T03:04:20,mike98,27.0,75,2affba40-9f86-4084-2f3d-56135f66ee65,0.0,NaN
69330,2017-01-29T05:14:29,lauriethompson,14.0,89,NaN,0.0,NaN


Wir sehen mehrere Felder, die als `NaN` ausgegeben werden. Um dies nun systematisch zu analysieren, wenden wir [verify_df](https://tdda.readthedocs.io/en/v1.0.31/constraints.html#tdda.constraints.verify_df) auf unseren neuen DataFrame an. Dabei gibt `passes` gibt die Anzahl der bestandenen, `failures` die Anzahl der fehlgeschlagenen Constraints zurück.

In [14]:
v = verify_df(new_df, "../../../data/iot_re_example.json")

In [15]:
v

In [16]:
v.passes

34

In [17]:
v.failures

3

Wir können uns auch anzeigen lassen, in welchen Spalten welche Constraints bestanden und fehlgeschlagen sind: 

In [18]:
print(str(v))

FIELDS:

timestamp: 0 failures  6 passes  type ✓  min_length ✓  max_length ✓  max_nulls ✓  no_duplicates ✓  rex ✓

username: 0 failures  5 passes  type ✓  min_length ✓  max_length ✓  max_nulls ✓  rex ✓

temperature: 1 failure  4 passes  type ✓  min ✓  max ✓  sign ✓  max_nulls ✗

heartrate: 0 failures  5 passes  type ✓  min ✓  max ✓  sign ✓  max_nulls ✓

build: 1 failure  5 passes  type ✓  min_length ✓  max_length ✓  max_nulls ✗  no_duplicates ✓  rex ✓

latest: 1 failure  4 passes  type ✓  min ✓  max ✓  sign ✓  max_nulls ✗

note: 0 failures  5 passes  type ✓  min_length ✓  max_length ✓  allowed_values ✓  rex ✓

SUMMARY:

Constraints passing: 34
Constraints failing: 3


Alternativ können wir uns diese Ergebnisse auch tabellarisch anzeigen lassen:

In [19]:
v.to_frame()

,field,failures,passes,type,min,min_length,max,max_length,sign,max_nulls,no_duplicates,allowed_values,rex
0,timestamp,0,6,True,NaN,True,NaN,True,NaN,True,True,NaN,True
1,username,0,5,True,NaN,True,NaN,True,NaN,True,NaN,NaN,True
2,temperature,1,4,True,True,NaN,True,NaN,True,False,NaN,NaN,NaN
3,heartrate,0,5,True,True,NaN,True,NaN,True,True,NaN,NaN,NaN
4,build,1,5,True,NaN,True,NaN,True,NaN,False,True,NaN,True
5,latest,1,4,True,True,NaN,True,NaN,True,False,NaN,NaN,NaN
6,note,0,5,True,NaN,True,NaN,True,NaN,NaN,NaN,True,True


## 6. Finden der fehlerhaften Zeilen

`tdda.constraints.pd.constraints.detect_df()` erkennt Datensätze des pandas DataFrame, die gegen eine der Einschränkungen in der bereitgestellten JSON-Datei verstoßen. Anschließend können wir über dem erstellten `PandasDetection`-Objekt die Funktion `detected()` aufrufen um uns die Zeilen ausgeben zu lassen, die fehlerhaft sind:

In [20]:
d = detect_df(new_df, "iot_example.json")

d.detected()

,n_failures
Index,
3,1
4,1
7,1
10,2
12,1
...,...
146385,1
146387,2
146391,2


Wir können uns alle fehlerhaften Datensätze anzeigen lassen, indem wir nur den Teil des Index von `new_df` verwenden, der auch in `d.detected()` vorkommt:

In [21]:
d_index = d.detected().index

In [22]:
new_df[new_df.index.isin(d_index)]

,timestamp,username,temperature,heartrate,build,latest,note
3,2017-01-01T12:02:09,eddierodriguez,28.0,76,NaN,0.0,update
4,2017-01-01T12:02:36,kenneth94,29.0,62,122f1c6a-403c-2221-6ed1-b5caa08f11e0,NaN,NaN
7,2017-01-01T12:04:35,scott28,16.0,76,7a60219f-6621-e548-180e-ca69624f9824,NaN,interval
10,2017-01-01T12:06:21,njohnson,NaN,63,e09b6001-125d-51cf-9c3f-9cb686c19d02,NaN,NaN
12,2017-01-01T12:07:41,jessica48,22.0,83,03e1a07b-3e14-412c-3a69-6b45bc79f81c,NaN,update
...,...,...,...,...,...,...,...
146385,2017-02-28T23:53:59,powelleric,20.0,86,152eda10-676a-069c-b664-19443f2c8081,NaN,test
146387,2017-02-28T23:54:50,jthompson,NaN,66,8da10303-fe49-e313-8fda-0d5e79ded054,NaN,update
146391,2017-02-28T23:57:21,aaronbecker,NaN,87,7e52f4a8-345c-5ee0-e515-b8c392213062,NaN,sleep
146393,2017-02-28T23:58:43,joelrusso,NaN,89,NaN,0.0,NaN


Alternativ können wir uns auch alle fehlerfreien Datensätze anzeigen lassen.

In [23]:
new_df[~new_df.index.isin(d_index)]

,timestamp,username,temperature,heartrate,build,latest,note
0,2017-01-01T12:00:23,michaelsmith,12.0,67,4e6a7805-8faa-2768-6ef6-eb3198b483ac,0.0,interval
1,2017-01-01T12:01:09,kharrison,6.0,78,7256b7b0-e502-f576-62ec-ed73533c9c84,0.0,wake
2,2017-01-01T12:01:34,smithadam,5.0,89,9226c94b-bb4b-a6c8-8e02-cb42b53e9c90,0.0,NaN
5,2017-01-01T12:03:04,bryanttodd,13.0,86,0897dbe5-9c5b-71ca-73a1-7586959ca198,0.0,interval
6,2017-01-01T12:03:51,andrea98,17.0,81,1c07ab9b-5f66-137d-a74f-921a41001f4e,1.0,NaN
...,...,...,...,...,...,...,...
146389,2017-02-28T23:56:05,kathy63,5.0,88,c2f76050-abd4-aee4-7bc0-3498325d0573,0.0,NaN
146390,2017-02-28T23:56:34,cookallison,16.0,84,f0b0c1f9-900b-276c-bca9-ac4d4ec4e88e,0.0,user
146392,2017-02-28T23:58:06,mcontreras,15.0,63,69e61a15-d2d0-47a7-1a27-e07b3eeeba10,0.0,NaN
146395,2017-02-28T23:59:48,grayjasmin,17.0,64,4911a589-3a15-4bbf-1de1-e5a69ab739da,1.0,update
